<div align="center">
    <br>
    <img alt="logo" src="https://avatars.githubusercontent.com/u/194089448?s=200&v=4" width="100"/>
    <p>
    Spectral - Adversarial AI Testing made easy
    </p>
    <p align="center">
       <a href="https://principled-intelligence.com/">Blog Article</a>&nbsp&nbsp | &nbsp&nbsp <a href="https://principled-intelligence.com">Principled Intelligence
        </a>
    </p>
    <hr/>
</div>

## Spectral: Evaluating a private RAG chatbot

> **New to Spectral?** Start with the quick-start notebook — set up a simple RAG chatbot and test it with Spectral, no private network setup needed.
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1qtkqy3oDb5_JufaUd9vEUvwgY2jTXYyt/view?usp=sharing)

---

**If you're here**, your AI system lives on private infrastructure — a corporate network, an air-gapped cluster, a private cloud subnet — somewhere that isn't reachable from the public internet. You need Spectral to test it, but you can't open inbound firewall rules, hand VPN credentials to a third party, stand up a public proxy, or rely on tunneling tools like ngrok — just to run an evaluation.

**[spectral-bridge](https://github.com/Principled-Intelligence/spectral-bridge)** solves this with a single **outbound** WebSocket connection from your network to Spectral's relay. Nothing inbound. No firewall changes. No new public infrastructure. Only the chatbot's text responses cross the boundary — nothing else on your network is reachable.

---
## What is Spectral?
**Spectral** is an AI evaluation platform. It simulates realistic users interacting with your live AI endpoint and scores every response on two main axes: *factual grounding* (are answers faithful to the knowledge base?) and *principle compliance* (do they respect your behavioural rules?). The result is an actionable report that shows exactly where your application falls short, with no manual test-writing required.

### What you will have at the end

- A locally running RAG chatbot that answers questions over a small Wikipedia corpus.
- A live spectral-bridge connection that lets Spectral reach that chatbot without a public URL.
- A reproducible baseline you can modify (KB size, chat model, system prompt) and re-evaluate.

test.svg


## Prerequisites

All credentials are loaded from **Colab Secrets** (`Runtime → Secrets` in the left sidebar). No `.env` files needed, nothing hardcoded.

| Secret | Required | What it's for |
|---|---|---|
| `OPENAI_API_KEY` | **Yes** | Text embeddings for the RAG index. Also usable as the chat LLM. |
| `ANTHROPIC_API_KEY` | Optional | Chat LLM backend if you switch the model to Claude. |
| `GOOGLE_API_KEY` | Optional | Chat LLM backend if you switch the model to Gemini. |
| `SPECTRAL_BRIDGE_API_KEY` | **Yes** | API key for [spectral-bridge](https://github.com/Principled-Intelligence/spectral-bridge) — obtained from the [Spectral dashboard](https://spectral.principled.app/) when creating an Internal target. |


### Step 1 — Create an Internal target in Spectral

spectral-bridge needs an API key tied to an **Internal** target in the Spectral dashboard. Do this before running any code.

> **→ Open the [Spectral dashboard](https://spectral.principled.app/) now**
>
> 1. Go to **My Targets → New target → Internal**
> 2. Insert a name for your target →
> 3. Copy the `SPECTRAL_BRIDGE_API_KEY` shown — **it will not be shown again**
> 4. In Colab: `Runtime → Secrets` → add a secret named `SPECTRAL_BRIDGE_API_KEY` and paste the value
> 5. **Come back here and run the next cell**


### Step 2 — Load Colab secrets

Pull API keys from Colab Secrets into environment variables. `simple-chatbot` and `litellm` read them at startup.

In [1]:
from google.colab import userdata
import os

# OPENAI_API_KEY is mandatory for embedding generation
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

optional_llm_api_keys = ["ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]
for key in optional_llm_api_keys:
    try:
        os.environ[key] = userdata.get(key)
    except userdata.SecretNotFoundError:
        pass

# spectral-bridge reads SPECTRAL_BRIDGE_API_KEY directly from the environment.
try:
  os.environ["SPECTRAL_BRIDGE_API_KEY"] = userdata.get("SPECTRAL_BRIDGE_API_KEY")
except:
  print('''
  SPECTRAL_BRIDGE_API_KEY not found. → Open https://spectral.principled.app/ with a browser now. Go to My Targets → New target → Internal. Insert a name for your target →
Copy the SPECTRAL_BRIDGE_API_KEY shown — it will not be shown again
In Colab: Runtime → Secrets → add a secret named SPECTRAL_BRIDGE_API_KEY and paste the value
''')
  raise


### Step 3 — Install dependencies

Installs:

- `simple-chatbot` — the RAG server (currently installed from a private GitHub repo; requires `GITHUB_TOKEN`)
- `spectral-bridge` — the relay client that connects to Spectral without a public URL
- `datasets` — streams the Wikipedia dataset from HuggingFace


In [2]:
import os

!pip install --force-reinstall -q "numpy>=2.0,<2.1" "pyarrow>=15" "datasets>=2.20" spectral-bridge "spectral-bridge[pass-through]" \
  git+https://github.com/Principled-Intelligence/simple-chatbot.git \
  requests python-dotenv > /dev/null 2>&1


### Step 4 — Configure your chatbot

This is the only cell you usually need to edit. Defaults: stream the first **300** documents from English Wikipedia, use `gpt-5.4-nano` as the chat model, set a Wikipedia-expert system prompt.

**Swap the knowledge base:** point `HF_DATASET` / `HF_CONFIG` at any HuggingFace dataset with a text column, then update `TEXT_FIELD` / `TITLE_FIELD` to match its schema.

**Swap the chat model:** set `CHAT_MODEL` to any litellm-compatible string (`anthropic/claude-sonnet-4-6`, `gemini/gemini-2.0-flash`, `openai/gpt-4o`, …). The corresponding API key must already be in Colab Secrets and loaded in Step 1.

**Swap the system prompt:** rewriting `SYSTEM_PROMPT` changes the chatbot's persona and rules — which is exactly the surface Spectral will probe.


In [3]:
# Configuration — edit these to switch KB, model, or prompt
HF_DATASET    = "wikimedia/wikipedia"
HF_CONFIG     = "20231101.en"   # required for wikimedia/wikipedia, set it to None if unnecessary for your selected dataset.
TEXT_FIELD    = "text"          # mandatory: column that contains the document body
TITLE_FIELD   = "title"         # optional: column used for filenames; set to None to use row index
HF_SPLIT      = "train"
KB_SIZE = 300
CHAT_MODEL    = "openai/gpt-5.4-nano"
EMBEDDING_MODEL = "openai/text-embedding-3-small"
PORT          = 8000
SYSTEM_PROMPT = "You are a Wikipedia expert that replies to user queries."

### Step 5 — Download the knowledge base

Streams `KB_SIZE` articles from HuggingFace and writes each one as a `.txt` file under `kbs/<dataset-name>/`. Streaming avoids a full dataset download — we only pay for the bytes we read.

`simple-chatbot` picks up this folder directly via `--docs-dir` and indexes it on startup. Supported file types: `.txt`, `.md`, `.pdf`, `.docx`.


In [4]:
from datasets import load_dataset

HF_NAME = HF_DATASET.split("/")[-1]
KB_DIR = f"kbs/{HF_NAME}"
import os
os.makedirs(KB_DIR, exist_ok=True)

print(f"Downloading first {KB_SIZE} documents from {HF_DATASET} (split={HF_SPLIT})...")

load_kwargs = dict(split=HF_SPLIT, streaming=True)
if HF_SPLIT is not None:
    dataset = load_dataset(HF_DATASET, HF_CONFIG, **load_kwargs)
else:
    dataset = load_dataset(HF_DATASET, **load_kwargs)

written = 0
for idx, row in enumerate(dataset):
    if written >= KB_SIZE:
        break
    # Filename: use TITLE_FIELD if set, else fall back to row index
    filename = str(row[TITLE_FIELD]).replace("/", "_").replace("\\", "_")[:100] if TITLE_FIELD is not None else f"doc_{idx}"
    body = row[TEXT_FIELD]
    with open(f"{KB_DIR}/{filename}.txt", "w") as f:
        f.write(body)
    written += 1

print(f"✓ Wrote {written} documents to {KB_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

✓ Wrote 300 documents to kbs/wikipedia


### Step 6 — Start the RAG chatbot server

Launches `simple-chatbot serve` as a background process. On startup it:

1. Scans `kbs/<dataset-name>/` for documents.
2. Chunks them (default: 500 tokens, 50-token overlap).
3. Embeds every chunk via OpenAI's `text-embedding-3-small`.
4. Stores the vectors in a local ChromaDB collection.
5. Binds a FastAPI server to `localhost:8000` exposing `POST /v1/chat/completions`.



In [5]:

import time

!nohup simple-chatbot serve \
  --docs-dir {KB_DIR} \
  --chat-model "{CHAT_MODEL}" \
  --system-prompt "{SYSTEM_PROMPT}" \
  --embedding-model "{EMBEDDING_MODEL}" \
  --port {PORT} \
  --log-level INFO \
  > /tmp/chatbot.log 2>&1 &

last = ""
while True:
    log = open("/tmp/chatbot.log").read()
    if "Starting HTTP server" in log:
        print("✓ Ready!")
        break
    last_line = log.strip().splitlines()[-1] if log.strip() else ""
    if last_line != last:
        print(last_line)
        last = last_line
    time.sleep(3)

✓ Ready!


### Step 7 — Smoke-test the endpoint

Sends a single chat request directly to the local chatbot (no public URL).
If this prints an answer grounded in the KB, the full pipeline is healthy.


In [6]:
LOCAL_URL = f"http://localhost:{PORT}/v1/chat/completions/"

import json
import requests


def send_message(user_message: str):
  print(f"👤 User: {user_message}")
  response = requests.post(
      LOCAL_URL,
      json={
          "messages": [
              {"role": "user", "content": user_message}
          ]
      },
      timeout=30,
  )
  response.raise_for_status()

  result = response.json()
  assistant_message = result["choices"][0]["message"]["content"]
  print(f"🤖 Assistant: {assistant_message}")


user_message = "Can you check in your provided documents when was Alabama founded?"
send_message(user_message)

print("-"*100)

user_message = "Name three of Andy Warhol's most famous works"
send_message(user_message)

print("-"*100)

user_message = "Briefly summarize one of Hercule Poirot's cases as described on his Wikipedia page"
send_message(user_message)



👤 User: Can you check in your provided documents when was Alabama founded?
🤖 Assistant: According to the provided document, **Alabama was admitted as the 22nd state on December 14, 1819**.
----------------------------------------------------------------------------------------------------
👤 User: Name three of Andy Warhol's most famous works
🤖 Assistant: Three of Andy Warhol’s most famous works are:

- **Campbell’s Soup Cans** (1962)  
- **Marilyn Diptych** (1962)  
- **Brillo Box** (1964)
----------------------------------------------------------------------------------------------------
👤 User: Briefly summarize one of Hercule Poirot's cases as described on his Wikipedia page
🤖 Assistant: One of Hercule Poirot’s earliest cases mentioned on his Wikipedia page is **_The Mysterious Affair at Styles_ (1916)**. After meeting his longtime friend **Captain Arthur Hastings**, Poirot solves the mystery—this is identified as the **first of his cases to be published**.


### Step 8 — What is spectral-bridge and how does it work?

[spectral-bridge](https://github.com/Principled-Intelligence/spectral-bridge) is a lightweight relay client that lets Spectral reach AI systems running on private networks — no public URL, no inbound firewall rules.

**How it works:**

1. `spectral-bridge` opens a persistent **outbound WebSocket** to `wss://bridge.spectral.principled.app/connect` — no port forwarding needed.
2. When Spectral runs an evaluation, it sends each probe as a message over that WebSocket.
3. `spectral-bridge` receives the message, forwards it as a normal HTTP POST to `http://localhost:8000/v1/chat/completions`, and sends the chatbot's response back over the same WebSocket.
4. Spectral receives the response as if it had called a public endpoint — but the chatbot never left your private network.

The API key (`SPECTRAL_BRIDGE_API_KEY`) was generated in Step 1 when you created the Internal target. spectral-bridge reads it automatically from the environment.


### Step 9 — Start spectral-bridge

Launches `spectral-bridge start` as a background process. It spawns a built-in `pass-through` adapter (which forwards HTTP to the chatbot) and connects to Spectral's relay.

In [7]:
!nohup spectral-bridge start \
  --relay-url wss://bridge.spectral.principled.app/connect \
  --adapter pass-through \
  --target http://localhost:{PORT} \
  > /tmp/bridge.log 2>&1 &

import time
print("Starting spectral-bridge... (wait for relay connection)")
time.sleep(5)
print("Bridge log so far:")
print(open("/tmp/bridge.log").read() or "(empty — still starting)")


Starting spectral-bridge... (wait for relay connection)
Bridge log so far:
[14:17:50] INFO     starting pass-through adapter, forwarding traffic from port 
                    8840 to target http://localhost:8000                        
[14:17:51] INFO     HTTP Request: GET http://127.0.0.1:8840/health "HTTP/1.1 200
                    OK"                                                         
           INFO     adapter ready                                               



### Step 10 — Verify the bridge connection

Poll the bridge log until it confirms a relay connection, then switch to the Spectral dashboard to run the built-in connectivity test.


In [8]:
import time

last = ""
while True:
    log = open("/tmp/bridge.log").read()
    if "connected" in log.lower():
        print("\u2713 spectral-bridge connected to relay!")
        break
    last_line = log.strip().splitlines()[-1] if log.strip() else ""
    if last_line != last:
        print(last_line)
        last = last_line
    time.sleep(2)


           INFO     adapter ready
✓ spectral-bridge connected to relay!


> **→ Switch back to the [Spectral dashboard](https://spectral.principled.app/)**
>
> Open your Internal target and click **"Test connection"** to confirm that Spectral can reach the chatbot through the relay.
>
> Once the connection shows as verified, come back here and run the smoke test below.


### Step 11 — (Optional) Export the knowledge base

Zips the `kbs/` folder and downloads it to your local machine. Handy if you want to upload the KB alongside your target in Spectral.


In [9]:
'''import shutil
from google.colab import files

# Zip the kbs folder
shutil.make_archive('/content/kbs', 'zip', '/content', 'kbs')

# Download
files.download('/content/kbs.zip')
'''

"import shutil\nfrom google.colab import files\n\n# Zip the kbs folder\nshutil.make_archive('/content/kbs', 'zip', '/content', 'kbs')\n\n# Download\nfiles.download('/content/kbs.zip')\n"

### Step 12 — Stop the server and bridge

Run this when you're done with the notebook. Stops the chatbot process and closes the spectral-bridge connection.


In [11]:
'''!pkill -f "simple-chatbot serve"
!pkill -f "spectral-bridge"
'''


## Next: run an evaluation in Spectral

> **→ Your chatbot is live on Spectral**
>
> Head to the [Spectral dashboard](https://spectral.principled.app/), open the Internal target you created in Step 1, and launch an evaluation.

Within a few minutes you'll have a scored report flagging principle violations and factuality gaps.

### Troubleshooting

- **`spectral-bridge` fails to connect** — `SPECTRAL_BRIDGE_API_KEY` is missing or invalid. Re-run Steps 1 and 2, making sure the key is saved to Colab Secrets.
- **Embedding errors during Step 5** — `OPENAI_API_KEY` is missing, expired, or out of quota. Text embeddings always go through OpenAI regardless of which chat model you picked.
- **Smoke test returns 502 or an empty response** — the chatbot is still indexing. Check `/tmp/chatbot.log` (Step 9) and retry once it logs `Starting HTTP server`.
- **"Test connection" fails in the dashboard** — check `/tmp/bridge.log` to confirm spectral-bridge is running and shows a connected status.
